# Week 5 Lecture: APIs and Data Formats for DevOps
**NET2008 DevOps · Algonquin College**

Weeks 3 and 4 were about writing Python that runs correctly and survives bad conditions. This session is about what that Python is *for*: talking to other systems. Almost everything in DevOps is one program asking another program for data, or telling it to change something, and the two programs agree on two things before they can do that:

1. a **data format** that both sides can read and write (JSON, XML, YAML), and
2. an **interface** that defines how to ask (an HTTP API, and in Python the `requests` library to call it).

Everything for this session lives in this notebook. There are no slides. Read the prose, run the cells, change the values, and watch what happens. A small practice API runs *inside* this notebook, so every example works offline and gives every student identical results. Cells that fail **on purpose** are wrapped in `try`/`except` so the notebook still runs top to bottom.

**Session plan (one break, after section 5)**

Subsections marked **REFERENCE** are not presented in class. The full detail stays here for you to read on your own, and the lab practises it.

## What We Will Cover

1. **Data, formats, and serialization**
   - Why data formats exist
   - Comparing JSON, XML, and YAML
   - Serialization and deserialization
   - Python types for structured data

2. **JSON**
   - Syntax and data types
   - `dumps()` and `loads()`: JSON strings
   - `dump()` and `load()`: JSON files
   - Formatting JSON for readability
   - Navigating nested data
   - Type changes and unsupported types
   - Handling `JSONDecodeError`
   - Practice exercises

3. **XML**
   - Elements, attributes, and text
   - Attributes versus child elements
   - Parsing and searching with `ElementTree`
   - Building and modifying XML
   - Namespaces
   - Handling `ParseError`
   - Converting XML into a Python dictionary
   - Practice exercises

4. **YAML**
   - Syntax, indentation, and comments
   - Reading and writing with PyYAML
   - Multiline strings, multiple documents, and anchors
   - Automatic type conversion
   - Safe parsing with `safe_load()`
   - Handling YAML errors
   - YAML in DevOps configuration files
   - The relationship between JSON and YAML
   - Practice exercises

5. **Choosing and converting formats**
   - Selecting the appropriate format
   - Converting between JSON, XML, and YAML

6. **APIs and HTTP fundamentals**
   - What an API is
   - URL structure
   - HTTP requests and responses
   - Methods: `GET`, `POST`, `PUT`, `PATCH`, and `DELETE`
   - Status codes
   - Request and response headers
   - Exploring the practice API

7. **The `requests` library**
   - Making HTTP requests
   - The `Response` object
   - Query parameters
   - Checking errors with `raise_for_status()`
   - Sending JSON data
   - Updating and deleting resources
   - Headers and content negotiation
   - Authentication
   - Timeouts and exception handling
   - Sessions
   - Safe retries
   - Pagination
   - Calling a public API

8. **Putting it together: an inventory client**
   - Building a reusable API client class
   - Combining data formats, API calls, and error handling
   - Automating an inventory task

**Where this sits in the course.** The curriculum lists *Data Formats: JSON, XML, YAML* and *Interfaces: REST APIs and the Requests library*. Week 6 (Git/GitHub) uses both ideas immediately, because GitHub is itself a REST API that returns JSON, and Weeks 7 to 13 (Docker, Kubernetes, Ansible, Terraform, CI/CD) are configured almost entirely in YAML or JSON.

---
## Setup: install, import, and start the practice API

Run the next three cells once, in order. You do not need to read the server code to follow the lecture; section 6 explains exactly what it does and why.

**Colab:** `requests` and `PyYAML` are already installed, so the first cell only confirms it. **Your own machine:** the first cell installs anything missing.

In [ ]:
import importlib, subprocess, sys

for module, pip_name in [("requests", "requests"), ("yaml", "pyyaml")]:
    try:
        importlib.import_module(module)
        print(f"{pip_name:<9} already installed")
    except ImportError:
        print(f"{pip_name:<9} missing, installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import requests, yaml
print("requests", requests.__version__, "| PyYAML", yaml.__version__, "| Python", sys.version.split()[0])

requests  already installed
pyyaml    already installed
requests 2.32.4 | PyYAML 6.0.3 | Python 3.13.15


In [ ]:
%%writefile mockapi.py
"""Mock REST API for the Week 5 lecture and lab (NET2008 DevOps).

A tiny device-inventory service built on the standard library only. It runs
inside the notebook in a background thread, so every example works offline and
every student sees exactly the same responses.

Endpoints (all under http://127.0.0.1:<port>):

    GET    /api/devices                list, paginated (?page=1&per_page=5, ?status=up)
    GET    /api/devices/<id>           one device (404 if unknown)
    POST   /api/devices                create (201; 400 on bad JSON; 422 on missing fields)
    PUT    /api/devices/<id>           replace (200)
    PATCH  /api/devices/<id>           partial update (200)
    DELETE /api/devices/<id>           delete (204)
    GET    /api/devices.xml            same inventory as XML
    GET    /api/devices.yaml           same inventory as YAML
    GET    /api/secure/ping            needs  Authorization: Bearer netops-token
    GET    /api/flaky                  503 twice, then 200 (resets via /api/reset)
    GET    /api/slow                   answers after 2 seconds
    GET    /api/limited                429 + Retry-After on the 3rd call in a row
    GET    /api/echo                   returns the query string, headers it received
    POST   /api/reset                  restore the seed data and all counters
"""
import copy
import json
import threading
import time
import xml.etree.ElementTree as ET
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

TOKEN = "netops-token"

SEED_DEVICES = [
    {"id": 1, "hostname": "OTT-CORE-01", "ip": "10.10.1.1", "type": "router", "ports": 8, "status": "up", "tags": ["core", "ospf"]},
    {"id": 2, "hostname": "OTT-CORE-02", "ip": "10.10.1.2", "type": "router", "ports": 8, "status": "up", "tags": ["core", "bgp"]},
    {"id": 3, "hostname": "OTT-SW-01", "ip": "10.10.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]},
    {"id": 4, "hostname": "OTT-SW-02", "ip": "10.10.2.2", "type": "switch", "ports": 48, "status": "down", "tags": ["access"]},
    {"id": 5, "hostname": "OTT-SW-03", "ip": "10.10.2.3", "type": "switch", "ports": 24, "status": "up", "tags": ["access", "lab"]},
    {"id": 6, "hostname": "TOR-FW-01", "ip": "10.20.1.1", "type": "firewall", "ports": 12, "status": "up", "tags": ["edge"]},
    {"id": 7, "hostname": "TOR-FW-02", "ip": "10.20.1.2", "type": "firewall", "ports": 12, "status": "down", "tags": ["edge"]},
    {"id": 8, "hostname": "TOR-SW-01", "ip": "10.20.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]},
    {"id": 9, "hostname": "MTL-RTR-01", "ip": "10.30.1.1", "type": "router", "ports": 4, "status": "up", "tags": ["branch"]},
    {"id": 10, "hostname": "MTL-SW-01", "ip": "10.30.2.1", "type": "switch", "ports": 24, "status": "up", "tags": ["branch"]},
    {"id": 11, "hostname": "MTL-AP-01", "ip": "10.30.3.1", "type": "access-point", "ports": 2, "status": "down", "tags": ["branch", "wifi"]},

]

_lock = threading.Lock()
_state = {}


def _reset():
    _state["devices"] = copy.deepcopy(SEED_DEVICES)
    _state["next_id"] = max(d["id"] for d in SEED_DEVICES) + 1
    _state["flaky_calls"] = 0
    _state["limited_calls"] = 0


_reset()


def devices_to_xml(devices):
    root = ET.Element("inventory", site="bytown")
    for d in devices:
        el = ET.SubElement(root, "device", id=str(d["id"]), type=d["type"])
        ET.SubElement(el, "hostname").text = d["hostname"]
        ET.SubElement(el, "ip").text = d["ip"]
        ET.SubElement(el, "ports").text = str(d["ports"])
        ET.SubElement(el, "status").text = d["status"]
        tags = ET.SubElement(el, "tags")
        for t in d["tags"]:
            ET.SubElement(tags, "tag").text = t
    return ET.tostring(root, encoding="unicode")


def devices_to_yaml(devices):
    lines = ["site: bytown", "devices:"]
    for d in devices:
        lines.append(f"  - id: {d['id']}")
        lines.append(f"    hostname: {d['hostname']}")
        lines.append(f"    ip: {d['ip']}")
        lines.append(f"    type: {d['type']}")
        lines.append(f"    ports: {d['ports']}")
        lines.append(f"    status: {d['status']}")
        lines.append("    tags: [" + ", ".join(d["tags"]) + "]")
    return "\n".join(lines) + "\n"


class Handler(BaseHTTPRequestHandler):
    server_version = "BytownMock/1.0"
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):  # keep the notebook output clean
        pass

    # ---- helpers -------------------------------------------------------
    def _send(self, code, body=None, ctype="application/json", headers=None):
        if body is None:
            payload = b""
        elif isinstance(body, (dict, list)):
            payload = json.dumps(body).encode("utf-8")
        elif isinstance(body, str):
            payload = body.encode("utf-8")
        else:
            payload = body
        self.send_response(code)
        if payload:
            self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(payload)))
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        if payload:
            try:
                self.wfile.write(payload)
            except (BrokenPipeError, ConnectionResetError):
                pass                     # the client gave up (for example after a timeout)

    def _error(self, code, message, headers=None):
        self._send(code, {"error": message, "status": code}, headers=headers)

    def _read_json(self):
        length = int(self.headers.get("Content-Length") or 0)
        raw = self.rfile.read(length) if length else b""
        try:
            return json.loads(raw.decode("utf-8")), None
        except (ValueError, UnicodeDecodeError):
            return None, "request body is not valid JSON"

    def _route(self):
        parsed = urlparse(self.path)
        return parsed.path.rstrip("/") or "/", parse_qs(parsed.query)

    def _device_id(self, path):
        parts = path.split("/")
        if len(parts) == 4 and parts[1] == "api" and parts[2] == "devices" and parts[3].isdigit():
            return int(parts[3])
        return None

    def _find(self, dev_id):
        for d in _state["devices"]:
            if d["id"] == dev_id:
                return d
        return None

    # ---- verbs ---------------------------------------------------------
    def do_GET(self):
        path, q = self._route()
        with _lock:
            if path == "/api/devices":
                items = list(_state["devices"])
                if "status" in q:
                    items = [d for d in items if d["status"] == q["status"][0]]
                if "type" in q:
                    items = [d for d in items if d["type"] == q["type"][0]]
                try:
                    page = int(q.get("page", ["1"])[0])
                    per_page = int(q.get("per_page", ["5"])[0])
                except ValueError:
                    return self._error(400, "page and per_page must be integers")
                if page < 1 or per_page < 1:
                    return self._error(400, "page and per_page must be >= 1")
                total = len(items)
                start = (page - 1) * per_page
                chunk = items[start:start + per_page]
                host = self.headers.get("Host", "127.0.0.1")
                nxt = None
                if start + per_page < total:
                    nxt = f"http://{host}/api/devices?page={page + 1}&per_page={per_page}"
                    if "status" in q:
                        nxt += f"&status={q['status'][0]}"
                return self._send(200, {"page": page, "per_page": per_page, "total": total, "next": nxt, "results": chunk})

            dev_id = self._device_id(path)
            if dev_id is not None:
                d = self._find(dev_id)
                return self._send(200, d) if d else self._error(404, f"device {dev_id} not found")

            if path == "/api/devices.xml":
                return self._send(200, devices_to_xml(_state["devices"]), "application/xml")
            if path == "/api/devices.yaml":
                return self._send(200, devices_to_yaml(_state["devices"]), "application/yaml")

            if path == "/api/secure/ping":
                auth = self.headers.get("Authorization", "")
                if not auth:
                    return self._error(401, "missing Authorization header", {"WWW-Authenticate": "Bearer"})
                if auth != f"Bearer {TOKEN}":
                    return self._error(403, "token not accepted")
                return self._send(200, {"pong": True, "user": "netops"})

            if path == "/api/flaky":
                _state["flaky_calls"] += 1
                if _state["flaky_calls"] <= 2:
                    return self._error(503, f"temporarily unavailable (call {_state['flaky_calls']})")
                return self._send(200, {"ok": True, "calls_needed": _state["flaky_calls"]})

            if path == "/api/limited":
                _state["limited_calls"] += 1
                if _state["limited_calls"] % 3 == 0:
                    return self._error(429, "slow down", {"Retry-After": "1"})
                return self._send(200, {"ok": True, "call": _state["limited_calls"]})

            if path == "/api/echo":
                return self._send(200, {"query": {k: v if len(v) > 1 else v[0] for k, v in q.items()},
                                        "user_agent": self.headers.get("User-Agent"),
                                        "accept": self.headers.get("Accept")})

        if path == "/api/slow":
            time.sleep(2)
            return self._send(200, {"ok": True, "waited": 2})
        return self._error(404, f"no route for GET {path}")

    def do_POST(self):
        path, _ = self._route()
        if path == "/api/reset":
            with _lock:
                _reset()
            return self._send(200, {"reset": True})
        if path == "/api/devices":
            body, err = self._read_json()
            if err:
                return self._error(400, err)
            if not isinstance(body, dict):
                return self._error(400, "body must be a JSON object")
            missing = [k for k in ("hostname", "ip", "type") if not body.get(k)]
            if missing:
                return self._error(422, "missing fields: " + ", ".join(missing))
            with _lock:
                dev = {"id": _state["next_id"], "hostname": body["hostname"], "ip": body["ip"], "type": body["type"],
                       "ports": body.get("ports", 0), "status": body.get("status", "up"), "tags": body.get("tags", [])}
                _state["next_id"] += 1
                _state["devices"].append(dev)
            return self._send(201, dev, headers={"Location": f"/api/devices/{dev['id']}"})
        return self._error(404, f"no route for POST {path}")

    def do_PUT(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for PUT {path}")
        body, err = self._read_json()
        if err:
            return self._error(400, err)
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            missing = [k for k in ("hostname", "ip", "type") if not isinstance(body, dict) or not body.get(k)]
            if missing:
                return self._error(422, "PUT replaces the whole record; missing: " + ", ".join(missing))
            d.clear()
            d.update({"id": dev_id, "hostname": body["hostname"], "ip": body["ip"], "type": body["type"],
                      "ports": body.get("ports", 0), "status": body.get("status", "up"), "tags": body.get("tags", [])})
            return self._send(200, d)

    def do_PATCH(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for PATCH {path}")
        body, err = self._read_json()
        if err:
            return self._error(400, err)
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            if not isinstance(body, dict):
                return self._error(400, "body must be a JSON object")
            for k, v in body.items():
                if k != "id":
                    d[k] = v
            return self._send(200, d)

    def do_DELETE(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for DELETE {path}")
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            _state["devices"].remove(d)
            return self._send(204)


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):   # no tracebacks in the notebook output
        pass


_server = None


def start(port=0):
    """Start the server in a daemon thread and return its base URL."""
    global _server
    if _server is None:
        _server = QuietServer(("127.0.0.1", port), Handler)
        threading.Thread(target=_server.serve_forever, daemon=True).start()
    return f"http://127.0.0.1:{_server.server_address[1]}"


def stop():
    global _server
    if _server is not None:
        _server.shutdown()
        _server.server_close()
        _server = None


def reset():
    with _lock:
        _reset()

Writing mockapi.py


In [ ]:
import mockapi

BASE = mockapi.start()          # starts the practice API in a background thread
print("practice API listening at", BASE)

# A first, tiny taste of what is coming. Do not worry about the details yet.
import requests
print(requests.get(f"{BASE}/api/devices/1").json())

practice API listening at http://127.0.0.1:37533
{'id': 1, 'hostname': 'OTT-CORE-01', 'ip': '10.10.1.1', 'type': 'router', 'ports': 8, 'status': 'up', 'tags': ['core', 'ospf']}


---
## 1. Data, formats, and serialization

### 1.1 Why formats exist

A Python dictionary lives in your program's memory, as objects with memory addresses that mean nothing to any other program. The moment data has to **leave** your program (saved to a file, sent to a device, posted to a web service, handed to another tool in a pipeline) it has to become a sequence of bytes that the receiver can turn back into something it understands.

- **Serialization** (also called *encoding*) turns an in-memory structure into text or bytes.
- **Deserialization** (also called *decoding* or *parsing*) turns that text or bytes back into an in-memory structure.

A **data format** is the agreement about what the text looks like. If both sides follow the agreement, a Python script can exchange the data without knowing anything about each other.

```
 your program                                 other system
 ------------                                 ------------
 dict / list   --serialize-->   text/bytes   --deserialize-->   its own objects
 dict / list   <--deserialize-- text/bytes   <--serialize---    its own objects
```

All three formats in this lecture are **text** formats. That choice costs a little speed and space, and buys something DevOps work depends on: a human can open the file or read the HTTP response and see what is going on.

### 1.2 Comparing the three

| Feature | JSON | XML | YAML |
|---|---|---|---|
| Full name | JavaScript Object Notation | Extensible Markup Language | YAML Ain’t Markup Language |
| Structure | Objects `{}` and arrays `[]` | Nested elements with tags and attributes | Indented mappings and lists; also supports `{}` and `[]` |
| Comments | Not supported | `<!-- comment -->` | `# comment` |
| Data types | String, number, Boolean, null, array, object | Element text and attribute values are text; schemas can define types | Strings, numbers, Booleans, null, sequences, and mappings; some parsers also recognize timestamps |
| Readability | Compact and familiar | Explicit but often verbose | Concise, with fewer structural symbols |
| Common pitfalls | Missing commas, double quotes required, no trailing commas | Mismatched tags and unescaped special characters | Incorrect indentation and unexpected automatic type conversion |
| Python support | Standard library: `json` | Standard library: `xml.etree.ElementTree` | Third-party package: `PyYAML` |
| Typical DevOps uses | REST API payloads, tool output, state files | NETCONF, SOAP, enterprise tools, build files | Ansible, Docker Compose, Kubernetes, CI/CD configuration |

**Whitespace matters differently:**
- **JSON:** Indentation improves readability but does not define structure.
- **XML:** Indentation generally improves readability, although whitespace within text content matters.
- **YAML:** Indentation defines nesting. Use spaces, not tabs, for indentation.

There is no universally best format. When working with an existing API or tool, use the format it requires. When designing your own solution, choose based on the data, readability, and available tooling.

### 1.3 The Python types that matter

Every format ends up as combinations of a small set of Python types. This table is the bridge between what you learned in Week 3 and what comes next.

| Python value | Becomes in JSON | Comes back as |
|---|---|---|
| `dict` | object `{...}` | `dict` |
| `list`, `tuple` | array `[...]` | `list` (the tuple is lost) |
| `str` | string `"..."` | `str` |
| `int`, `float` | number | `int` or `float` |
| `True`, `False` | `true`, `false` | `True`, `False` |
| `None` | `null` | `None` |
| `set`, `bytes`, `datetime`, your own classes | **not supported** | needs help (section 2.7) |

Keep that last row in mind. Most "my JSON export crashed" bugs are one of those four things hiding inside a dictionary.

---
## 2. JSON

### 2.1 Syntax

JSON has exactly six kinds of value and a short list of rules.

| JSON value | Example | Notes |
|---|---|---|
| object | `{"hostname": "sw01", "ports": 48}` | an unordered set of `"key": value` pairs; keys are **always strings** |
| array | `["core", "ospf"]` | an ordered list of values |
| string | `"OTT-SW-01"` | **double quotes only** |
| number | `48`, `-3`, `2.5`, `1e3` | no quotes, no leading zeros, no `0x` hex |
| boolean | `true`, `false` | lowercase |
| null | `null` | lowercase |

Rules that cause most real-world errors:

- strings and keys use **double** quotes, never single quotes;
- **no trailing commas** after the last item;
- **no comments** of any kind;
- `True`, `None` and `NaN` (Python spellings) are not JSON.

JSON values nest to any depth, which is how a flat-looking text file describes a whole inventory.

### 2.2 `dumps()` and `loads()`: strings; `dump()` and `load()`: files

JSON is a **text format**. To work with JSON data in Python, we convert between JSON text and Python objects, such as dictionaries and lists.

- **Serialization:** Python object → JSON text
- **Deserialization:** JSON text → Python object

The `json` module provides four functions:

| Function | What it does
|---|---|
| `json.dumps(obj)` | Converts a Python object into a JSON string (serialization)
| `json.loads(text)` | Parses a JSON string into a Python object (deserialization)
| `json.dump(obj, file)` | Writes a Python object as JSON into an open file (serialization)
| `json.load(file)` | Reads JSON from an open file into a Python object (deserialization)

**How to remember the names:**
- **dump**: send data **out of Python** as JSON text through serialization.
- **load**: bring JSON data **into Python** through deserialization.
- When sending data over a network, you generally cannot send a Python object directly. You serialize it into a format the receiving system understands, such as JSON.
- The trailing **s** means **string**. Without it, you work with an open file.

#### Working with strings

```python
import json

device = {"hostname": "OTT-CORE-01", "ports": 8}

# Serialize: Python dictionary → JSON string
text = json.dumps(device)
print(type(text))       # <class 'str'>

# Deserialize: JSON string → Python dictionary
data = json.loads(text)
print(type(data))       # <class 'dict'>

In [ ]:
import json

device = {"hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48, "uplink": True, "last_error": None}

# Serialize: Python dictionary → JSON text in a file
with open("device.json", "w", encoding="utf-8") as file:
    json.dump(device, file)

# Read the file as text to see the serialized JSON
with open("device.json", "r", encoding="utf-8") as file:
    encoded = file.read()

print(type(encoded).__name__, "->", encoded)

# Deserialize: JSON text in a file → Python dictionary
with open("device.json", "r", encoding="utf-8") as file:
    decoded = json.load(file)

print(type(decoded).__name__, "->", decoded)
print("round trip equal?", decoded == device)

str -> {"hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48, "uplink": true, "last_error": null}
dict -> {'hostname': 'OTT-SW-01', 'ip': '10.10.2.1', 'ports': 48, 'uplink': True, 'last_error': None}
round trip equal? True


In [ ]:
device = {"hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48, "uplink": True, "last_error": None}

encoded = json.dumps(device)
print(type(encoded).__name__, "->", encoded)

decoded = json.loads(encoded)
print(type(decoded).__name__, "->", decoded)
print("round trip equal?", decoded == device)

str -> {"hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48, "uplink": true, "last_error": null}
dict -> {'hostname': 'OTT-SW-01', 'ip': '10.10.2.1', 'ports': 48, 'uplink': True, 'last_error': None}
round trip equal? True


Notice what `dumps` did to the Python spellings: `True` became `true` and `None` became `null`. The output is JSON text, not Python source code, and pasting it into a Python file would raise a `NameError`.

### 2.3 Making JSON readable

By default `dumps` produces one long line with no extra spaces, which is what machines want. Three options change that.

| Option | Effect |
|---|---|
| `indent=2` | pretty-print with 2 spaces per level |
| `sort_keys=True` | alphabetical keys, so two runs produce identical text (excellent for diffs in Git) |
| `ensure_ascii=False` | write non-ASCII characters as themselves instead of `é` escapes |

In [ ]:
device = {"hostname": "OTT-SW-01", "location": "Ottawa, Québec", "ports": 48, "ip": "10.10.2.1"}

print(json.dumps(device))
print()
print(json.dumps(device, ensure_ascii=False))
print()
print(json.dumps(device, indent=2, sort_keys=True, ensure_ascii=False))

{"hostname": "OTT-SW-01", "location": "Ottawa, Qu\u00e9bec", "ports": 48, "ip": "10.10.2.1"}

{"hostname": "OTT-SW-01", "location": "Ottawa, Québec", "ports": 48, "ip": "10.10.2.1"}

{
  "hostname": "OTT-SW-01",
  "ip": "10.10.2.1",
  "location": "Ottawa, Québec",
  "ports": 48
}


### 2.4 Navigating nested data

Once parsed, JSON is plain Python, so everything from Week 3 applies: indexing, slicing, loops, comprehensions. What changes is that the structure is **someone else's**, so you cannot assume a key exists.

A reliable habit: use `[...]` when the key is *required* and its absence should be an error, and use `.get(key, default)` when it is *optional*.

In [ ]:
inventory = json.loads("""
{
  "site": "bytown",
  "devices": [
    {
      "hostname": "OTT-CORE-01",
      "type": "router",
      "interfaces": [
        {
          "name": "Gi0/0",
          "ip": "10.10.1.1",
          "up": true
        },
        {
          "name": "Gi0/1",
          "ip": "10.10.9.1",
          "up": false
        }
      ]
    },
    {
      "hostname": "OTT-SW-01",
      "type": "switch",
      "interfaces": [
        {
          "name": "Fa0/1",
          "ip": null,
          "up": true
        }
      ]
    }
  ]
}
""")

# Drill down one level at a time.
first = inventory["devices"][0]
print(first["hostname"], "has", len(first["interfaces"]), "interfaces")
print("second interface:", first["interfaces"][1]["name"], "up =", first["interfaces"][1]["up"])

# Comprehension over nested data: every interface that is down, with its device.
down = [(d["hostname"], i["name"]) for d in inventory["devices"] for i in d["interfaces"] if not i["up"]]
print("down interfaces:", down)

# Optional keys: .get() with a default instead of a KeyError.
for d in inventory["devices"]:
    print(d["hostname"], "| location:", d.get("location", "unknown"))

OTT-CORE-01 has 2 interfaces
second interface: Gi0/1 up = False
down interfaces: [('OTT-CORE-01', 'Gi0/1')]
OTT-CORE-01 | location: unknown
OTT-SW-01 | location: unknown


When data is nested several levels deep, repeated `.get()` calls get ugly. A small helper that walks a path and returns a default is worth keeping in your toolkit.

In [ ]:
def dig(data, *path, default=None):
    """Follow keys/indexes through nested dicts and lists; return default if any step is missing."""
    current = data
    for step in path:
        try:
            current = current[step]
        except (KeyError, IndexError, TypeError):
            return default
    return current

print(dig(inventory, "devices", 0, "interfaces", 1, "ip"))
print(dig(inventory, "devices", 5, "hostname", default="no such device"))
print(dig(inventory, "devices", 1, "interfaces", 0, "ip", default="n/a"))   # value is null -> None

10.10.9.1
no such device
None


### 2.5 What does not survive the round trip

JSON is smaller than Python, so a few things change on the way through. None of these raise an error, which is why they are worth seeing once.

Three things to remember:

- **Tuples come back as lists.** If it matters, convert after loading.
- **Dictionary keys come back as strings.** `{1: "one"}` returns as `{"1": "one"}`, so `back[1]` raises `KeyError`. This is a classic bug when device IDs are used as keys.

In [ ]:
original = {
    "tuple_value": (10, 20, 30),
    "int_key": {1: "one", 2: "two"},
    "nested_none": [None, True, False],
}
back = json.loads(json.dumps(original))

for key in original:
    print(f"{key:} > before: {original[key]!r:<45} after: {back[key]!r}")

print()
print("tuple became list :", back["tuple_value"] == [10, 20, 30], "| equal to original tuple?", back["tuple_value"] == original["tuple_value"])
print("int keys became str:", list(back["int_key"].keys()))

tuple_value > before: (10, 20, 30)                                  after: [10, 20, 30]
int_key > before: {1: 'one', 2: 'two'}                          after: {'1': 'one', '2': 'two'}
nested_none > before: [None, True, False]                           after: [None, True, False]

tuple became list : True | equal to original tuple? False
int keys became str: ['1', '2']


`json.dumps` raises `TypeError` for anything outside the table in section 1.3. The two ways to teach it are:

1. **`default=` function** for encoding: called for every object `json` does not recognize; return something it *does* recognize.
2. **`object_hook=` function** for decoding: called with every parsed object (dict); return whatever you want in its place.

A `datetime` is the most common case in DevOps data (timestamps on every log entry and API response).

In [ ]:
from datetime import datetime, timezone

record = {"hostname": "OTT-SW-01", "checked_at": datetime(2026, 10, 1, 13, 30, tzinfo=timezone.utc), "tags": {"core", "lab"}}

try:
    json.dumps(record)
except TypeError as e:
    print("TypeError:", e)

TypeError: Object of type datetime is not JSON serializable


In order to fix above error you can use `to_jsonable` callback function:

In [ ]:
def to_jsonable(obj):
    """Called by json.dumps for any object it cannot encode itself."""
    if isinstance(obj, datetime):
        return obj.isoformat()                 # "2026-10-01T13:30:00+00:00"
    if isinstance(obj, (set, frozenset)):
        return sorted(obj)                     # sets are unordered, so sort for stable output
    if hasattr(obj, "to_dict"):
        return obj.to_dict()                   # your own classes, like the Week 4 devices
    raise TypeError(f"{type(obj).__name__} is not JSON serializable")

text = json.dumps(record, default=to_jsonable, indent=2)
print(text)

# Going back: the text has plain strings, so we convert the ones we know are timestamps.
loaded = json.loads(text)
loaded["checked_at"] = datetime.fromisoformat(loaded["checked_at"])
print(type(loaded["checked_at"]).__name__, loaded["checked_at"].year)

{
  "hostname": "OTT-SW-01",
  "checked_at": "2026-10-01T13:30:00+00:00",
  "tags": [
    "core",
    "lab"
  ]
}
datetime 2026


In [ ]:
# Your Week 4 classes plug straight in through to_dict().
class Switch:
    def __init__(self, hostname, ip, ports):
        self.hostname, self.ip, self.ports = hostname, ip, ports
    def to_dict(self):
        return {"type": type(self).__name__, "hostname": self.hostname, "ip": self.ip, "ports": self.ports}

fleet = [Switch("OTT-SW-01", "10.10.2.1", 48), Switch("OTT-SW-02", "10.10.2.2", 24)]
print(json.dumps(fleet, default=to_jsonable))

[{"type": "Switch", "hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48}, {"type": "Switch", "hostname": "OTT-SW-02", "ip": "10.10.2.2", "ports": 24}]


### 2.6 Malformed JSON and `JSONDecodeError`

> **REFERENCE** Read it on your own; the lab practises it.

`json.loads` is strict. When the text is not valid JSON it raises `json.JSONDecodeError`, which is a subclass of `ValueError` and carries the position of the problem. Everything from last week exception handling applies: **catch the specific exception, report something useful, and keep going if the program can.**

Here are the four mistakes that produce almost every `JSONDecodeError`.

In [ ]:
bad_documents = {
    "single quotes":   "{'hostname': 'sw01'}",
    "trailing comma":  '{"hostname": "sw01", "ports": 48,}',
    "comment":         '{"hostname": "sw01"  // core switch\n}',
    "python literals": '{"up": True, "error": None}',
    "empty body":      "",
}

for label, doc in bad_documents.items():
    try:
        json.loads(doc)
    except json.JSONDecodeError as e:
        print(f"{label:<16} line {e.lineno} col {e.colno}: {e.msg}")

single quotes    line 1 col 2: Expecting property name enclosed in double quotes
trailing comma   line 1 col 33: Illegal trailing comma before end of object
comment          line 1 col 22: Expecting ',' delimiter
python literals  line 1 col 8: Expecting value
empty body       line 1 col 1: Expecting value


In [ ]:
def load_json_text(text, source="input"):
    """Parse JSON or raise a ValueError that names where the data came from."""
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"{source}: invalid JSON at line {e.lineno}, column {e.colno} ({e.msg})") from e

try:
    load_json_text('{"hostname": "sw01",}', source="inventory.json")
except ValueError as e:
    print(e)
    print("original cause:", type(e.__cause__).__name__)      # chaining from Week 4, section 8.10

inventory.json: invalid JSON at line 1, column 20 (Illegal trailing comma before end of object)
original cause: JSONDecodeError


---
## 3. XML

### 3.1 Syntax

XML describes data as a **tree of elements**. Where JSON uses braces, XML uses **tags**: every element has an opening tag, content, and a closing tag. For example, the following `json` data:

```json
{
  "inventory": {
    "site": "bytown",
    "device": {
      "hostname": "sw01",
      "ports": 48
    }
  }
}
```

is equal to the `XML` data below:
```xml
<?xml version="1.0" encoding="UTF-8"?>
<inventory>
  <site>bytown</site>
  <device id="3">
    <hostname>sw01</hostname>
    <ports>48</ports>
  </device>
</inventory>
```

The vocabulary:

| Term | Example | Meaning |
|---|---|---|
| **element** | `<ports>48</ports>` | a start tag, content, and an end tag |
| **tag** | `ports` | the element's name |
| **attribute** | `id="3"` | a name/value pair inside the start tag for identity and metadata; the value is always quoted |
| **text** | `48` | the characters between the tags |
| **root element** | `<inventory>` | the single outermost element; a document has exactly one |
| **declaration** | `<?xml version="1.0"?>` | optional first line stating version and encoding |

A document is **well-formed** if it follows the syntax rules. Violating any of them makes the whole document unreadable (there is no "mostly valid" XML):

- exactly one root element;
- every start tag has a matching end tag, and tags nest properly (`<a><b></a></b>` is illegal);
- attribute values are quoted;
- tag names are case-sensitive (`<Port>` is not `</port>`);
- the five special characters in text must be escaped: `&lt;` `&gt;` `&amp;` `&quot;` `&apos;`.

### 3.2 Attribute or child element?

The same fact can be written either way, and XML does not tell you which to choose:

```xml
<device id="3" type="switch"/>                           <!-- attributes only -->

<device><id>3</id><type>switch</type></device>           <!-- child elements only -->

<device id="3">
    <type>switch</type>
</device>           
```

A common convention: use **attributes for identity and metadata** (`id`, `type`, `lang`) and **elements for the data itself** (`hostname`, `ports`). Attributes cannot repeat or nest, so anything that can have several values (a list of tags) has to be elements. You will meet both styles in real documents, so your code has to read both.

### 3.3 Parsing with `ElementTree`

Python's standard library parses XML with `xml.etree.ElementTree`, almost always imported as `ET`. After parsing you hold an **Element**, which has four things worth knowing:

| Attribute / method | Gives you |
|---|---|
| `el.tag` | the element name |
| `el.attrib` | a dict of its attributes |
| `el.text` | the text directly inside it (or `None`) |
| `list(el)` or `for child in el` | its child elements |

Use `ET.fromstring(text)` for a string and `ET.parse(path)` for a file (`ET.parse(path).getroot()` gives the root element).

In [ ]:
import xml.etree.ElementTree as ET

xml_text = """<?xml version="1.0" encoding="UTF-8"?>
<inventory site="bytown">
  <device id="1" type="router">
    <hostname>OTT-CORE-01</hostname>
    <ip>10.10.1.1</ip>
    <ports>8</ports>
    <status>up</status>
    <tags><tag>core</tag><tag>ospf</tag></tags>
  </device>
  <device id="3" type="switch">
    <hostname>OTT-SW-01</hostname>
    <ip>10.10.2.1</ip>
    <ports>48</ports>
    <status>up</status>
    <tags><tag>access</tag></tags>
  </device>
  <device id="4" type="switch">
    <hostname>OTT-SW-02</hostname>
    <ip>10.10.2.2</ip>
    <ports>48</ports>
    <status>down</status>
    <tags><tag>access</tag></tags>
  </device>
</inventory>"""

root = ET.fromstring(xml_text)

print("root tag       :", root.tag)
print("root attributes:", root.attrib)
print("children       :", [child.tag for child in root])
print("first device   :", root[0].tag, root[0].attrib)
print("its children   :", [child.tag for child in root[0]])
print("hostname text  :", root[0][0].text)

root tag       : inventory
root attributes: {'site': 'bytown'}
children       : ['device', 'device', 'device']
first device   : device {'id': '1', 'type': 'router'}
its children   : ['hostname', 'ip', 'ports', 'status', 'tags']
hostname text  : OTT-CORE-01


### 3.4 Finding things

> **REFERENCE** Read it on your own; the lab practises it.


Indexing like `root[0][0]` breaks the moment someone adds an element. Searching by name is safer.

| Method | Returns | Notes |
|---|---|---|
| `el.find("tag")` | first matching **direct child**, or `None` | |
| `el.findall("tag")` | list of all matching direct children | |
| `el.findtext("tag", default="")` | the child's text directly | skips the `None` check |
| `el.iter("tag")` | every matching element at **any depth** | |
| `el.get("attr", default)` | an attribute's value | same as `el.attrib.get(...)` |

`find` and `findall` also understand a limited **XPath** syntax:

| Path | Meaning |
|---|---|
| `"device"` | direct children named `device` |
| `".//tag"` | `tag` anywhere below here |
| `"device[@type='switch']"` | `device` whose `type` attribute is `switch` |
| `"device/hostname"` | `hostname` inside `device` |

In [ ]:
# Every device, with attributes and child text pulled out by name.
for dev in root.findall("device"):
    print(dev.get("id"), dev.get("type"), dev.findtext("hostname"), dev.findtext("ip"))

print()
print("switches only :", [d.findtext("hostname") for d in root.findall("device[@type='switch']")])
print("all tags      :", [t.text for t in root.iter("tag")])
print("down hostnames:", [d.findtext("hostname") for d in root.findall("device") if d.findtext("status") == "down"])

1 router OTT-CORE-01 10.10.1.1
3 switch OTT-SW-01 10.10.2.1
4 switch OTT-SW-02 10.10.2.2

switches only : ['OTT-SW-01', 'OTT-SW-02']
all tags      : ['core', 'ospf', 'access', 'access']
down hostnames: ['OTT-SW-02']


In [ ]:
# Text is ALWAYS a string. Convert before doing arithmetic or comparisons.
ports_as_text = [d.findtext("ports") for d in root.findall("device")]
print("raw text      :", ports_as_text)
print("sorted as text:", sorted(ports_as_text))                 # '48' < '8' alphabetically!
print("converted     :", sorted(int(p) for p in ports_as_text))
print("total ports   :", sum(int(p) for p in ports_as_text))

raw text      : ['8', '48', '48']
sorted as text: ['48', '48', '8']
converted     : [8, 48, 48]
total ports   : 104


> **Common pitfall.** `find()` returns `None` when nothing matches, and `None.text` raises `AttributeError`. For optional elements use `findtext("tag", default="")`, or test `if el is not None`. Do not write `if el:`: an element with no children is *falsy* even though it exists.

In [ ]:
dev = root.find("device")
print("missing element, naive    :", dev.find("location"))
try:
    dev.find("location").text
except AttributeError as e:
    print("AttributeError            :", e)
print("missing element, findtext :", repr(dev.findtext("location", default="unknown")))
print("empty element is falsy    :", bool(dev.find("status")), "(but it exists:", dev.find("status") is not None, ")")

missing element, naive    : None
AttributeError            : 'NoneType' object has no attribute 'text'
missing element, findtext : 'unknown'
empty element is falsy    : False (but it exists: True )


/tmp/ipykernel_6863/1864068356.py:8: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  print("empty element is falsy    :", bool(dev.find("status")), "(but it exists:", dev.find("status") is not None, ")")


### 3.5 Building and modifying XML

> **REFERENCE** Read it on your own; the lab practises it.

Creating a document is the reverse: build the tree with `ET.Element` and `ET.SubElement`, then serialize.

| Call | Effect |
|---|---|
| `ET.Element("tag", attr="v")` | a new element (use for the root) |
| `ET.SubElement(parent, "tag", attr="v")` | a new element **already attached** to `parent` |
| `el.text = "..."` | set the text (must be a string) |
| `el.set("attr", "v")` | set an attribute |
| `parent.remove(child)` | detach a child |
| `ET.tostring(el, encoding="unicode")` | serialize to a `str` (without `encoding="unicode"` you get `bytes`) |
| `ET.indent(tree)` | pretty-print in place (Python 3.9+) |

In [ ]:
report = ET.Element("report", generated="2026-10-01")

for hostname, state in [("OTT-SW-01", "up"), ("OTT-SW-02", "down")]:
    check = ET.SubElement(report, "check", device=hostname)
    ET.SubElement(check, "state").text = state
    ET.SubElement(check, "latency_ms").text = str(12.5 if state == "up" else 0)   # text must be str

ET.indent(report)                                    # pretty-print in place
print(ET.tostring(report, encoding="unicode"))

<report generated="2026-10-01">
  <check device="OTT-SW-01">
    <state>up</state>
    <latency_ms>12.5</latency_ms>
  </check>
  <check device="OTT-SW-02">
    <state>down</state>
    <latency_ms>0</latency_ms>
  </check>
</report>


In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET

# Safely resolve a working directory even if the current working directory was deleted
try:
    out_dir = Path.cwd()
except FileNotFoundError:
    # Fallback if current working directory has been deleted
    out_dir = Path("/tmp")
    os.chdir(out_dir)

# Modify an existing tree: mark OTT-SW-02 as up, remove the router, then write the file.
for dev in root.findall("device"):
    if dev.findtext("hostname") == "OTT-SW-02":
        dev.find("status").text = "up"
    if dev.get("type") == "router":
        root.remove(dev)

tree = ET.ElementTree(root)
ET.indent(tree)
tree.write(out_dir / "inventory.xml", encoding="utf-8", xml_declaration=True)
print((out_dir / "inventory.xml").read_text(encoding="utf-8"))

<?xml version='1.0' encoding='utf-8'?>
<inventory site="bytown">
  <device id="3" type="switch">
    <hostname>OTT-SW-01</hostname>
    <ip>10.10.2.1</ip>
    <ports>48</ports>
    <status>up</status>
    <tags>
      <tag>access</tag>
    </tags>
  </device>
  <device id="4" type="switch">
    <hostname>OTT-SW-02</hostname>
    <ip>10.10.2.2</ip>
    <ports>48</ports>
    <status>up</status>
    <tags>
      <tag>access</tag>
    </tags>
  </device>
</inventory>


Special characters are escaped for you when you set `.text`, and un-escaped when you read it. You never write `&amp;` by hand in Python.

In [ ]:
el = ET.Element("note")
el.text = 'ports < 24 & status = "down"'
serialized = ET.tostring(el, encoding="unicode")
print(serialized)
print(ET.fromstring(serialized).text)           # round trip restores the original characters

<note>ports &lt; 24 &amp; status = "down"</note>
ports < 24 & status = "down"


### 3.6 Namespaces

> **REFERENCE** Read it on your own; the lab practises it.

Large XML vocabularies (NETCONF, SOAP, SVG) share documents by prefixing every tag with a **namespace**: a URI that identifies which vocabulary an element belongs to. The URI is only a unique name. Nothing is fetched from it.

```xml
<interfaces xmlns="urn:ietf:params:xml:ns:yang:ietf-interfaces">
  <interface><name>Gi0/0</name><enabled>true</enabled></interface>
</interfaces>
```

`ElementTree` stores the namespace **inside the tag name** in curly braces, so a plain `find("interface")` finds nothing. Pass a `namespaces` mapping and use a prefix of your own choosing.

In [ ]:
netconf_reply = """
<interfaces xmlns="urn:ietf:params:xml:ns:yang:ietf-interfaces">
  <interface><name>Gi0/0</name><enabled>true</enabled></interface>
  <interface><name>Gi0/1</name><enabled>false</enabled></interface>
</interfaces>
"""
ns_root = ET.fromstring(netconf_reply)

print("tag as stored   :", ns_root.tag)
print("plain find      :", ns_root.find("interface"))                       # None: no namespace given

ns = {"if": "urn:ietf:params:xml:ns:yang:ietf-interfaces"}               # prefix is yours to choose
for interface in ns_root.findall("if:interface", ns):
    name = interface.findtext("if:name", namespaces=ns)
    enabled = interface.findtext("if:enabled", namespaces=ns) == "true"      # text -> real boolean
    print(f"{name}: enabled={enabled}")

tag as stored   : {urn:ietf:params:xml:ns:yang:ietf-interfaces}interfaces
plain find      : None
Gi0/0: enabled=True
Gi0/1: enabled=False


### 3.7 Malformed XML and `ParseError`

> **REFERENCE** Read it on your own; the lab practises it.

Because XML is all-or-nothing, a single mistake raises `xml.etree.ElementTree.ParseError`, with a `.position` attribute `(line, column)`.

In [ ]:
bad_xml = {
    "unclosed tag":     "<device><hostname>sw01</device>",
    "mismatched case":  "<Device></device>",
    "unquoted attr":    "<device id=3/>",
    "raw ampersand":    "<note>R&D lab</note>",
    "two roots":        "<a/><b/>",
    "empty":            "",
}
for label, doc in bad_xml.items():
    try:
        ET.fromstring(doc)
    except ET.ParseError as e:
        print(f"{label:<16} {e}")

unclosed tag     mismatched tag: line 1, column 24
mismatched case  mismatched tag: line 1, column 10
unquoted attr    not well-formed (invalid token): line 1, column 11
raw ampersand    not well-formed (invalid token): line 1, column 9
two roots        junk after document element: line 1, column 4
empty            no element found: line 1, column 0


### 3.8 From XML to a Python dict

> **REFERENCE** Read it on your own; the lab practises it.

Most of your code will be easier if XML is converted to the dicts and lists you already know. A short recursive function (recursion from Week 3, section 10.7) does it for the regular structures that tools generate.

In [ ]:
def element_to_dict(el):
    """Convert an Element to plain Python data. Attributes are prefixed with '@'. Repeated tags become lists."""
    result = {f"@{k}": v for k, v in el.attrib.items()}
    for child in el:
        value = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(value)
        else:
            result[child.tag] = value
    text = (el.text or "").strip()
    if not result:
        return text                         # leaf element: just its text
    if text:
        result["#text"] = text
    return result

fresh_root = ET.fromstring(xml_text)
as_dict = element_to_dict(fresh_root)
print(json.dumps(as_dict, indent=2)[:620], "...")

{
  "@site": "bytown",
  "device": [
    {
      "@id": "1",
      "@type": "router",
      "hostname": "OTT-CORE-01",
      "ip": "10.10.1.1",
      "ports": "8",
      "status": "up",
      "tags": {
        "tag": [
          "core",
          "ospf"
        ]
      }
    },
    {
      "@id": "3",
      "@type": "switch",
      "hostname": "OTT-SW-01",
      "ip": "10.10.2.1",
      "ports": "48",
      "status": "up",
      "tags": {
        "tag": "access"
      }
    },
    {
      "@id": "4",
      "@type": "switch",
      "hostname": "OTT-SW-02",
      "ip": "10.10.2.2",
      "ports": "48",
      "statu ...


This function shows the main difficulty of mapping XML onto JSON-style data: a single `<tag>` and a list of one `<tag>` look identical in XML, so a generic converter has to guess. When the shape matters (it usually does), write the extraction against the **specific** document you are handling, using `findall`, rather than a generic converter.

> **Security note.** `ElementTree` is not safe against maliciously crafted XML such as "billion laughs" (an entity that expands exponentially) when you parse documents from untrusted sources. For data from outside your organization, install and use the `defusedxml` package, which is a drop-in replacement: `import defusedxml.ElementTree as ET`.

---
## 4. YAML

### 4.1 Syntax

YAML expresses the same data model as JSON (mappings, sequences, scalars) but replaces braces and quotes with **indentation**, which makes it the easiest of the three to read. That is why tools whose main input is a file a human edits (Ansible, Docker Compose, Kubernetes, GitHub Actions) chose it.

```yaml
# a comment
site: bytown                  # mapping: key, colon, SPACE, value
active: true
last_error: null
devices:                      # a key whose value is a list
  - hostname: OTT-CORE-01     # "- " starts a sequence item
    ports: 8
    tags: [core, ospf]        # inline ("flow") list, JSON-like
  - hostname: OTT-SW-01
    ports: 48
    tags:
      - access
```

Is equivalent to the following `JSON` data:
```json
{
  "site": "bytown",
  "active": true,
  "last_error": null,
  "devices": [
    {
      "hostname": "OTT-CORE-01",
      "ports": 8,
      "tags": ["core", "ospf"]
    },
    {
      "hostname": "OTT-SW-01",
      "ports": 48,
      "tags": ["access"]
    }
  ]
}
```

| Construct | Syntax | Python result |
|---|---|---|
| mapping | `key: value` | `dict` |
| sequence | `- item` on its own line, or `[a, b]` | `list` |
| string | `hello`, `"hello"`, `'hello'` | `str` |
| number | `48`, `2.5` | `int`, `float` |
| boolean | `true` / `false` | `bool` |
| null | `null` or `~` or empty | `None` |
| comment | `# text` | ignored |

The rules that matter:

- **indentation is structure.** Use spaces, never tabs. Two spaces per level is the convention. Sibling items must line up exactly.
- a colon **must be followed by a space** (`ports:48` is a single string, not a mapping).
- quotes are *optional* for most strings, and *required* for some (section 4.4).

### 4.2 Reading and writing with PyYAML

PyYAML is a third-party package (`pip install pyyaml`, imported as `yaml`). Its four main functions parallel `json`:

| Task | Call |
|---|---|
| YAML text/file to Python | `yaml.safe_load(text_or_file)` |
| several documents in one stream | `yaml.safe_load_all(...)` |
| Python to YAML text | `yaml.safe_dump(obj)` |
| Python to YAML file | `yaml.safe_dump(obj, file)` |

Two options on `safe_dump` you will use constantly: `sort_keys=False` (keep your insertion order instead of alphabetizing) and `default_flow_style=False` (always use the indented block style, which is the readable one).

In [ ]:
import yaml

yaml_text = """
# Bytown inventory
site: bytown
active: true
last_error: null
devices:
  - hostname: OTT-CORE-01
    ports: 8
    tags: [core, ospf]
  - hostname: OTT-SW-01
    ports: 48
    tags:
      - access
"""

data = yaml.safe_load(yaml_text)          # Deserialization
print(data)
print(type(data["devices"]).__name__, "|", data["devices"][1]["tags"], "|", data["last_error"])

text_back = yaml.safe_dump(data, sort_keys=False) # Serialization
print(text_back)

{'site': 'bytown', 'active': True, 'last_error': None, 'devices': [{'hostname': 'OTT-CORE-01', 'ports': 8, 'tags': ['core', 'ospf']}, {'hostname': 'OTT-SW-01', 'ports': 48, 'tags': ['access']}]}
list | ['access'] | None
site: bytown
active: true
last_error: null
devices:
- hostname: OTT-CORE-01
  ports: 8
  tags:
  - core
  - ospf
- hostname: OTT-SW-01
  ports: 48
  tags:
  - access



In [ ]:
config = {
    "site": "bytown",
    "polling": {"interval_seconds": 30, "timeout_seconds": 5},
    "devices": [
        {"hostname": "OTT-SW-01", "ports": 48, "tags": ["access", "lab"]},
        {"hostname": "OTT-SW-02", "ports": 24, "tags": []},
    ],
    "notify": None,
}

with (out_dir / "config.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, default_flow_style=False) # File serialization: obj > file

with (out_dir / "config.yaml").open(encoding="utf-8") as f:
    print("round trip equal?", yaml.safe_load(f) == config)              # Deserialized file: text > obj

round trip equal? True


### 4.3 Multi-line strings, multiple documents, anchors

> **REFERENCE** Read it on your own; the lab practises it.

A few features you will see in real files.

**Multi-line strings.** `|` keeps line breaks exactly (scripts, certificates, banners). `>` folds lines into one paragraph (long descriptions).

**Multiple documents.** A line containing only `---` starts a new document in the same file. Kubernetes manifests commonly combine several objects this way.

**Anchors and aliases.** `&name` marks a value, `*name` reuses it, and `<<: *name` merges a mapping into another. It removes copy-paste in files with repeated settings (Docker Compose and CI files lean on this).

In [ ]:
doc = """
banner: |
  Authorized use only.
  All activity is logged.
summary: >
  This description is written over
  several lines but loads as
  one line of text.
"""
parsed = yaml.safe_load(doc)
print(repr(parsed["banner"]))
print(repr(parsed["summary"]))

'Authorized use only.\nAll activity is logged.\n'
'This description is written over several lines but loads as one line of text.\n'


In [ ]:
multi = """
kind: Namespace
name: lab
---
kind: Service
name: web
port: 80
"""
for document in yaml.safe_load_all(multi):
    print(document)

{'kind': 'Namespace', 'name': 'lab'}
{'kind': 'Service', 'name': 'web', 'port': 80}


In [ ]:
anchors = """
defaults: &defaults          # &defaults names this mapping
  timeout: 5
  retries: 3
  protocol: https

switch_api:
  <<: *defaults              # merge in everything from defaults...
  retries: 5                 # ...and override one value
  host: 10.10.2.1

router_api:
  <<: *defaults
  host: 10.10.1.1
"""
parsed = yaml.safe_load(anchors)
print(parsed["switch_api"])
print(parsed["router_api"])

{'timeout': 5, 'retries': 5, 'protocol': 'https', 'host': '10.10.2.1'}
{'timeout': 5, 'retries': 3, 'protocol': 'https', 'host': '10.10.1.1'}


### 4.4 The type-guessing trap

In JSON, `"48"` is a string and `48` is a number, because the quotes say so. In YAML, **you usually leave the quotes off, so the parser guesses the type from what the value looks like.** Its guesses are right for most data and wrong in exactly the places that hurt. PyYAML implements YAML 1.1, whose rules include:

| You wrote | PyYAML loads | What you probably meant |
|---|---|---|
| `enabled: yes` / `on` | `True` | maybe the word "yes" |
| `country: no` | `False` | Norway, the country code `NO` |
| `version: 3.10` | `3.1` (float) | the string `"3.10"` |
| `zip: 01234` | `668` (octal!) | the string `"01234"` |
| `port: 8080` | `8080` (int) | correct |
| `mac: 12:34:56` | `45296` (base-60 integer) | a MAC fragment |
| `started: 2026-10-01` | a `datetime.date` | a date, or the string |

**The fix is simple: quote any value that must be a string.** When a config value comes from someone else, print `type()` of anything surprising.

In [ ]:
risky = """
enabled: yes
country: no
version: 3.10
zip: 01234
started: 2026-10-01
quoted_version: "3.10"
quoted_country: "no"
"""
loaded = yaml.safe_load(risky)
for key, value in loaded.items():
    print(f"{key:<15} {value!r:<22} {type(value).__name__}")

enabled         True                   bool
country         False                  bool
version         3.1                    float
zip             668                    int
started         datetime.date(2026, 10, 1) date
quoted_version  '3.10'                 str
quoted_country  'no'                   str


### 4.5 `safe_load`, not `load`

> **REFERENCE** Read it on your own; the lab practises it.

YAML can carry **tags** that ask the parser to construct arbitrary Python objects. A loader that honours them can be made to run code from a document, so `yaml.load()` on untrusted text is a remote-code-execution vulnerability. `safe_load` only builds plain data (dicts, lists, strings, numbers, booleans, `None`, dates), and refuses everything else.

**Rule: always `yaml.safe_load` and `yaml.safe_dump`.** If a linter or tutorial shows bare `yaml.load(f)`, replace it.

In [ ]:
suspicious = "result: !!python/object/apply:os.getcwd []"        # asks the loader to call a function

try:
    yaml.safe_load(suspicious)
except yaml.YAMLError as e:
    print(type(e).__name__, "-> refused to build a Python object")

ConstructorError -> refused to build a Python object


### 4.6 Errors

> **REFERENCE** Read it on your own; the lab practises it.

PyYAML raises a subclass of `yaml.YAMLError`. Parser errors carry a `problem_mark` with the line and column. The most common causes are indentation mistakes and tabs.

In [ ]:
bad_yaml = {
    "tab indentation":  "devices:\n\t- sw01",
    "bad indentation":  "devices:\n  - hostname: sw01\n   ports: 48",
    "no space after :":  "ports:48",
    "unclosed bracket": "tags: [core, ospf",
}
for label, doc in bad_yaml.items():
    try:
        result = yaml.safe_load(doc)
        print(f"{label:<17} parsed without error (!): {result!r}")
    except yaml.YAMLError as e:
        mark = getattr(e, "problem_mark", None)
        where = f"line {mark.line + 1}, column {mark.column + 1}" if mark else "unknown position"
        print(f"{label:<17} {type(e).__name__} at {where}")

tab indentation   ScannerError at line 2, column 1
bad indentation   ParserError at line 3, column 4
no space after :  parsed without error (!): 'ports:48'
unclosed bracket  ParserError at line 1, column 18


Notice that `"ports:48"` did **not** fail. With no space after the colon it is not a mapping at all, just the single string `"ports:48"`, and the loader is happy. YAML's flexibility means *an unexpected result* is as likely as an error. Check the shape of anything you load, exactly as with JSON.

### 4.7 YAML you will meet in later weeks

> **REFERENCE** Read it on your own; the lab practises it.

Each of these is just a Python dict after `safe_load`. Parsing them now shows how little new there is to learn: the formats are the same, only the vocabulary changes.

In [ ]:
compose = yaml.safe_load("""
services:
  web:
    image: nginx:1.27
    ports: ["8080:80"]
    environment:
      MODE: production
  db:
    image: postgres:16
    volumes:
      - dbdata:/var/lib/postgresql/data
volumes:
  dbdata: {}
""")

print("services :", list(compose["services"]))
print("web image:", compose["services"]["web"]["image"])
print("web ports:", compose["services"]["web"]["ports"], type(compose["services"]["web"]["ports"][0]).__name__)

services : ['web', 'db']
web image: nginx:1.27
web ports: ['8080:80'] str


In [ ]:
workflow = yaml.safe_load("""
name: ci
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Run tests
        run: pytest -q
""")

print("top-level keys:", list(workflow))
print("trigger key   :", [k for k in workflow if k is True or k == "on"])      # 'on' loads as True in YAML 1.1!
steps = workflow["jobs"]["test"]["steps"]
print("step names    :", [s.get("name", s.get("uses")) for s in steps])

top-level keys: ['name', True, 'jobs']
trigger key   : [True]
step names    : ['actions/checkout@v4', 'Run tests']


Look at the `trigger key` line: GitHub Actions' `on:` key was loaded as the boolean `True`, because it is one of the YAML 1.1 boolean spellings from section 4.4. Real tools know this and handle it; it is also a reminder of why the type-guessing trap matters.

### 4.8 JSON is (almost) YAML

> **REFERENCE** Read it on your own; the lab practises it.

JSON is valid YAML for practical purposes, so `yaml.safe_load` happily reads JSON. This makes conversion between the two a one-liner in either direction.

In [ ]:
json_text = '{"hostname": "OTT-SW-01", "ports": 48, "tags": ["access", "lab"]}'

print("YAML parser reads JSON:", yaml.safe_load(json_text))
print()
print("JSON -> YAML:")
print(yaml.safe_dump(json.loads(json_text), sort_keys=False))
print("YAML -> JSON:")
print(json.dumps(yaml.safe_load("hostname: OTT-SW-01\nports: 48\ntags: [access, lab]\n"), indent=2))

YAML parser reads JSON: {'hostname': 'OTT-SW-01', 'ports': 48, 'tags': ['access', 'lab']}

JSON -> YAML:
hostname: OTT-SW-01
ports: 48
tags:
- access
- lab

YAML -> JSON:
{
  "hostname": "OTT-SW-01",
  "ports": 48,
  "tags": [
    "access",
    "lab"
  ]
}


---
## 5. Choosing and converting formats

### 5.1 Which one should I use?

You rarely choose freely: **the system on the other end decides.** When you do get to choose:

| Situation | Reach for | Reason | Examples |
|---|---|---|---|
| Calling or building a web API | **JSON** | Widely supported, compact, strict syntax | Requesting weather data; sending device information to a REST API |
| A config file people edit by hand | **YAML** | Supports comments and readable nesting | Ansible playbooks; Docker Compose files |
| A config file a program edits | **JSON** | Explicit types; easy to generate correctly | Saving application settings |
| Talking to a an older enterprise service | **XML** | Required by the protocol or service | Configuring a router through NETCONF |
| Saving program state for the next run | **JSON** | Portable; easy to compare with consistent formatting and `sort_keys=True` | Saving the last processed job ID; recording completed tasks |
| Tabular data for a spreadsheet | **CSV** | Supported by Python’s `csv` module and Excel | Exporting device inventory; saving student grades |

Other formats you will meet include:

- **INI:** simple configuration sections and key-value pairs, handled by `configparser`.
- **TOML:** configuration files such as `pyproject.toml`, read using `tomllib` in Python 3.11+.

All support reading and writing through suitable libraries, though their APIs differ. Python’s built-in `tomllib` supports **reading only**.

In [ ]:
# Because every format becomes dicts and lists, converting is "load with one library, dump with another".
def convert(text, target):
    """Convert text from 'json' to 'json', 'yaml' and 'xml-flat' (a simple tag-per-key XML for flat dicts)."""
    data = json.loads(text) # deserialize to get data obj

    if target == "json":
        return json.dumps(data, indent=2)
    if target == "yaml":
        return yaml.safe_dump(data, sort_keys=False)
    if target == "xml":
        root = ET.Element("device")
        for key, value in data.items():
            ET.SubElement(root, key).text = str(value)          # note: types are lost, everything is text
        ET.indent(root)
        return ET.tostring(root, encoding="unicode")
    raise ValueError(f"unsupported target format: {target}")

src = '{"hostname": "OTT-SW-01", "ip": "10.10.2.1", "ports": 48}'
for target in ("yaml", "xml", "json"):
    print(f"--- JSON -> {target}")
    print(convert(src, target))

--- JSON -> yaml
hostname: OTT-SW-01
ip: 10.10.2.1
ports: 48

--- JSON -> xml
<device>
  <hostname>OTT-SW-01</hostname>
  <ip>10.10.2.1</ip>
  <ports>48</ports>
</device>
--- JSON -> json
{
  "hostname": "OTT-SW-01",
  "ip": "10.10.2.1",
  "ports": 48
}


---
## 6. APIs and HTTP fundamentals

### 6.1 What an API is

An **API** (Application Programming Interface) is a defined way for one program to use another. A **web API** does the same across a network: your program sends a request to a URL, a server does some work, and sends back a response.

**REST** (Representational State Transfer) is the dominant style for web APIs. A REST API models the world as **resources** (devices, users, repositories, containers), gives each resource a **URL**, and uses standard **HTTP methods** to act on it. The representation that travels is usually JSON.

| Idea | In practice |
|---|---|
| Resource | a *thing*: a device, a list of devices |
| URL | its address: `/api/devices/3` |
| Method | the verb: `GET` it, `POST` a new one, `DELETE` it |
| Representation | the data format: JSON, sometimes XML or YAML |
| Stateless | every request carries everything the server needs (including credentials); the server remembers nothing between calls |

Other styles exist: SOAP (XML envelopes/repesentation, older enterprise services), GraphQL (one endpoint, the client describes the shape it wants), and gRPC (binary, between services). REST over HTTP with JSON is what the rest of this course, and nearly every DevOps tool you will use (GitHub, Docker, Kubernetes, Terraform Cloud, Jenkins, Zabbix), exposes.

### 6.2 Anatomy of a URL

```
https://api.example.com:8443/api/devices?status=down&page=2#top
\___/   \_____________/\___/\___________/ \______________/ \_/
scheme       host      port     path          query       fragment
```

| Part | Meaning |
|---|---|
| scheme | protocol: `http` or `https` (always prefer `https`: encrypted) |
| host | which server (`127.0.0.1` means "this computer") |
| port | which service on that server; defaults to 80 (http) and 443 (https) |
| path | which resource |
| query | `key=value` filters, joined with `&`, after a `?` |
| fragment | a position inside the page; never sent to the server |

The standard library's `urllib.parse` takes URLs apart and builds them safely. Query values must be **URL-encoded** (spaces, `&` and `=` inside a value would otherwise break the structure), and `requests` does this for you, as you will see.

In [ ]:
from urllib.parse import urlparse, parse_qs, urlencode, urljoin

url = "https://api.example.com:8443/api/devices?status=down&page=2#top"
parts = urlparse(url)

print("scheme  :", parts.scheme)
print("host    :", parts.hostname)
print("port    :", parts.port)
print("path    :", parts.path)
print("query   :", parts.query, "->", parse_qs(parts.query))
print("fragment:", parts.fragment)
print()
print("build a query string:", urlencode({"name": "core router", "tag": "a&b", "page": 2}))
print("join a path         :", urljoin("https://api.example.com/api/", "devices/3"))

scheme  : https
host    : api.example.com
port    : 8443
path    : /api/devices
query   : status=down&page=2 -> {'status': ['down'], 'page': ['2']}
fragment: top

build a query string: name=core+router&tag=a%26b&page=2
join a path         : https://api.example.com/api/devices/3


### 6.3 Anatomy of a request and a response

HTTP is plain text sent over a TCP connection. Every exchange is a **request** followed by a **response**, and both have the same three parts.

```
REQUEST                                          RESPONSE
GET /api/devices/3 HTTP/1.1                      HTTP/1.1 200 OK
Host: 127.0.0.1:8000                             Content-Type: application/json
Accept: application/json                         Content-Length: 118
Authorization: Bearer netops-token
                                                 {"id": 3, "hostname": "OTT-SW-01", ...}
(blank line, then an optional body)              (blank line, then the body)
\_____________/ start line                       \_______________/ status line
\______________/ headers                         \_____________/ headers
```

The next cell shows this is not a metaphor: it opens a raw TCP socket and sends exactly those bytes, with no HTTP library at all. After this, `requests` is just a convenience.

In [ ]:
import socket

port = int(BASE.rsplit(":", 1)[1])

raw_request = (
    "GET /api/devices/3 HTTP/1.1\r\n"
    f"Host: 127.0.0.1:{port}\r\n"
    "Accept: application/json\r\n"
    "Connection: close\r\n"
    "\r\n"                                    # blank line = end of headers
)

with socket.create_connection(("127.0.0.1", port), timeout=5) as sock:
    sock.sendall(raw_request.encode("ascii"))
    chunks = []
    while chunk := sock.recv(4096):
        chunks.append(chunk)

print(b"".join(chunks).decode("utf-8"))

HTTP/1.1 200 OK
Server: BytownMock/1.0 Python/3.13.15
Date: Sun, 04 Oct 2026 23:09:48 GMT
Content-Type: application/json
Content-Length: 120

{"id": 3, "hostname": "OTT-SW-01", "ip": "10.10.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]}


### 6.4 Methods

The method says *what you want to do* to the resource at the URL. Two properties decide how carefully you can call it:

- **safe** means it does not change anything on the server, or in other words the request is intended only to read information;
- **idempotent** means calling it ten times has the same effect as calling it once.

| Method | Meaning | CRUD | Has body? | Safe | Idempotent |
|---|---|---|---|---|---|
| `GET` | read a resource or a list | Read | no | yes | yes |
| `POST` | create a new resource (server picks the ID) | Create | yes | no | **no** |
| `PUT` | replace a resource completely | Update | yes | no | yes |
| `PATCH` | change some fields of a resource | Update | yes | no | usually |
| `DELETE` | remove a resource | Delete | no | no | yes |
| `HEAD` | like `GET` but headers only | | no | yes | yes |

Idempotency is not trivia. If a `POST /devices` times out, you do not know whether the device was created, and **retrying could create it twice**. A `GET` or `PUT` can be retried blindly. We return to this in section 7.

### 6.5 Status codes

The response starts with a three-digit **status code**. The first digit is the class, and the class tells you whose fault it is and what to do.

| Class | Meaning | What your code should do |
|---|---|---|
| **1xx** | informational | rarely seen; ignore |
| **2xx** | success | use the response |
| **3xx** | redirect | follow it (`requests` does for `GET`) |
| **4xx** | **client** error: your request is wrong | fix the request; **do not retry** (except 408 and 429) |
| **5xx** | **server** error: their side failed | may be temporary; retry with a delay |

The codes you will see weekly:

| Code | Name | Typical meaning |
|---|---|---|
| 200 | OK | request worked; body has the data |
| 201 | Created | `POST` worked; `Location` header gives the new URL |
| 204 | No Content | worked; there is deliberately no body (`DELETE`) |
| 301 / 302 | Moved | resource lives at another URL (`Location` header) |
| 400 | Bad Request | malformed request (broken JSON, bad parameter) |
| 401 | Unauthorized | **who are you?** missing or invalid credentials |
| 403 | Forbidden | **I know who you are, and no.** credentials fine, no permission |
| 404 | Not Found | no such resource |
| 405 | Method Not Allowed | that URL does not support that verb |
| 409 | Conflict | clashes with current state (duplicate name) |
| 422 | Unprocessable Entity | JSON is valid but the content is not (missing field) |
| 429 | Too Many Requests | rate limited; wait for `Retry-After` seconds |
| 500 | Internal Server Error | the server crashed |
| 502 / 503 / 504 | Bad Gateway / Unavailable / Gateway Timeout | server or something in front of it is down or overloaded; usually temporary |

`401` versus `403` is one of the most common points of confusion: 401 means *authentication* failed (prove who you are), 403 means *authorization* failed (you are not allowed).

### 6.6 Headers

> **REFERENCE** Read it on your own; the lab practises it.

Headers are `Name: value` lines that describe the request or response. You will read and set these:

| Header | Direction | Purpose |
|---|---|---|
| `Content-Type` | both | format of the **body being sent**: `application/json`, `application/xml` |
| `Accept` | request | format the client **wants back** |
| `Authorization` | request | credentials: `Bearer <token>` or `Basic <base64>` |
| `User-Agent` | request | who is calling; some APIs insist on one |
| `Location` | response | URL of a created or moved resource |
| `Retry-After` | response | seconds to wait before trying again (with 429 or 503) |
| `Content-Length` | both | size of the body in bytes |

Header names are case-insensitive. `requests` stores them in a dictionary that honours that.

---
## 7. The `requests` library

### 7.1 Why `requests`

Python can make HTTP calls with the standard library (`urllib.request`, and the raw socket you just saw), but it takes a page of code to do what `requests` does in a line: encoding parameters, parsing JSON, following redirects, handling compression, keeping connections open. `requests` is the third-party library nearly every Python DevOps script uses, and its design is deliberate: **one function per HTTP method**, each returning a **Response** object.

```python
requests.get(url, params=None, headers=None, timeout=None, ...)
requests.post(url, json=None, data=None, headers=None, timeout=None, ...)
requests.put(...)   requests.patch(...)   requests.delete(...)
```

In [ ]:
r = requests.get(f"{BASE}/api/devices/3", timeout=5)
print(r)                       # <Response [200]>
print(r.json())

<Response [200]>
{'id': 3, 'hostname': 'OTT-SW-01', 'ip': '10.10.2.1', 'type': 'switch', 'ports': 48, 'status': 'up', 'tags': ['access']}


### 7.2 The Response object

Everything the server sent is on the object you get back.

| Attribute | Gives you |
|---|---|
| `r.status_code` | integer: `200`, `404`, ... |
| `r.ok` | `True` if the status is below 400 |
| `r.reason` | the text that goes with the code: `"OK"`, `"Not Found"` |
| `r.headers` | case-insensitive dict of response headers |
| `r.text` | body decoded to a `str` |
| `r.content` | body as raw `bytes` (images, archives) |
| `r.json()` | body parsed as JSON (raises if it is not JSON) |
| `r.url` | the final URL that was requested, after encoding and redirects |
| `r.elapsed` | time between sending the request and receiving the headers |
| `r.request` | the request that was sent (headers, body) |

In [ ]:
r = requests.get(f"{BASE}/api/devices/3", timeout=5)

print("status_code :", r.status_code)
print("ok          :", r.ok)
print("reason      :", r.reason)
print("url         :", r.url)
print("elapsed     :", f"{r.elapsed.total_seconds() * 1000:.1f} ms")
print("Content-Type:", r.headers["Content-Type"])
print("content-type:", r.headers["content-type"], "(same header: names are case-insensitive)")
print()
print("text   :", r.text)
print("content:", r.content[:40], "...", type(r.content).__name__)
print("json() :", type(r.json()).__name__, r.json()["hostname"])

status_code : 200
ok          : True
reason      : OK
url         : http://127.0.0.1:44841/api/devices/3
elapsed     : 4.2 ms
Content-Type: application/json
content-type: application/json (same header: names are case-insensitive)

text   : {"id": 3, "hostname": "OTT-SW-01", "ip": "10.10.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]}
content: b'{"id": 3, "hostname": "OTT-SW-01", "ip":' ... bytes
json() : dict OTT-SW-01


### 7.3 Query parameters

Never build query strings by hand with an f-string. Pass a dict to `params=` and `requests` encodes it correctly (spaces, `&`, non-ASCII characters) and adds the `?` for you.

In [ ]:
r = requests.get(f"{BASE}/api/devices", params={"status": "down", "per_page": 10}, timeout=5)

print("final URL:", r.url)
body = r.json()
print(body)
print("keys     :", list(body))
print("total    :", body["total"])
for device in body["results"]:
    print(f'  {device["id"]:>2}  {device["hostname"]:<12} {device["status"]}')

final URL: http://127.0.0.1:37533/api/devices?status=down&per_page=10
{'page': 1, 'per_page': 10, 'total': 3, 'next': None, 'results': [{'id': 4, 'hostname': 'OTT-SW-02', 'ip': '10.10.2.2', 'type': 'switch', 'ports': 48, 'status': 'down', 'tags': ['access']}, {'id': 7, 'hostname': 'TOR-FW-02', 'ip': '10.20.1.2', 'type': 'firewall', 'ports': 12, 'status': 'down', 'tags': ['edge']}, {'id': 11, 'hostname': 'MTL-AP-01', 'ip': '10.30.3.1', 'type': 'access-point', 'ports': 2, 'status': 'down', 'tags': ['branch', 'wifi']}]}
keys     : ['page', 'per_page', 'total', 'next', 'results']
total    : 3
   4  OTT-SW-02    down
   7  TOR-FW-02    down
  11  MTL-AP-01    down


In [ ]:
# The echo endpoint shows what the server actually received, including awkward characters.
r = requests.get(f"{BASE}/api/echo", params={"name": "core router", "filter": "a&b=c", "tag": ["x", "y"]}, timeout=5)
print(r.url)
print(r.json()["query"])

http://127.0.0.1:44841/api/echo?name=core+router&filter=a%26b%3Dc&tag=x&tag=y
{'name': 'core router', 'filter': 'a&b=c', 'tag': ['x', 'y']}


### 7.4 Errors are not exceptions (unless you ask)

This is the single most important thing to understand about `requests`: **a 404 or a 500 does not raise an exception.** The call succeeded in the networking sense (the server answered); the *answer* just happened to be an error. Code that skips the status check carries on with the error body as if it were data.

In [ ]:
r = requests.get(f"{BASE}/api/devices/999", timeout=5)

print("status :", r.status_code, r.reason)
print("ok     :", r.ok)
print("body   :", r.json())                 # an error document, not a device

# The bug: treating it as a device
device = r.json()
print("hostname via naive code:", device.get("hostname"))      # silently None

status : 404 Not Found
ok     : False
body   : {'error': 'device 999 not found', 'status': 404}
hostname via naive code: None


Two ways to handle this properly:

1. **Check explicitly** when the status carries meaning your program should act on (a 404 might simply mean "does not exist yet, create it").
2. **`r.raise_for_status()`** when any 4xx or 5xx means failure: it raises `requests.HTTPError` and does nothing on a success. Add it right after the call and the rest of the function can assume success.

In [ ]:
def get_device(device_id):
    """Return the device dict, or None if it does not exist. Any other failure raises."""
    r = requests.get(f"{BASE}/api/devices/{device_id}", timeout=5)
    if r.status_code == 404:
        return None                         # an expected, meaningful outcome
    return r.json()

print(get_device(3)["hostname"])
print(get_device(999))

# What raise_for_status() looks like when it fires:
r = requests.get(f"{BASE}/api/devices/999", timeout=5)
try:
    r.raise_for_status()
except requests.HTTPError as e:
    print("HTTPError:", e)
    print("response still available:", e.response.status_code, e.response.json())

OTT-SW-01
None
HTTPError: 404 Client Error: Not Found for url: http://127.0.0.1:37533/api/devices/999
response still available: 404 {'error': 'device 999 not found', 'status': 404}


### 7.5 Sending data: `POST` with `json=`

Two parameters carry a request body, and the difference matters:

| Parameter | Sends | `Content-Type` set automatically |
|---|---|---|
| `json=obj` | `obj` serialized as JSON | `application/json` |
| `data=dict` | form fields, like an HTML form | `application/x-www-form-urlencoded` |
| `data="string"` | the string as is | none (you set it) |

For a JSON API, always use `json=`. Do **not** call `json.dumps` yourself and pass it to `data=`; you would lose the header and the server may reject it.

In [ ]:
import requests

def reset_api():
    """Restore the practice API to its starting state."""
    requests.post(f"{BASE}/api/reset", timeout=5).raise_for_status()

reset_api()
print("practice API reset; total devices:", requests.get(f"{BASE}/api/devices", timeout=5).json()["total"])

practice API reset; total devices: 11


In [ ]:
reset_api()

new_device = {"hostname": "OTT-AP-09", "ip": "10.10.3.9", "type": "access-point", "ports": 2, "tags": ["wifi"]}
r = requests.post(f"{BASE}/api/devices", json=new_device, timeout=5)

print("status  :", r.status_code, r.reason)
print("Location:", r.headers.get("Location"))
print("body    :", r.json())
print("we sent Content-Type:", r.request.headers["Content-Type"])
print("we sent body        :", r.request.body)

status  : 201 Created
Location: /api/devices/12
body    : {'id': 12, 'hostname': 'OTT-AP-09', 'ip': '10.10.3.9', 'type': 'access-point', 'ports': 2, 'status': 'up', 'tags': ['wifi']}
we sent Content-Type: application/json
we sent body        : b'{"hostname": "OTT-AP-09", "ip": "10.10.3.9", "type": "access-point", "ports": 2, "tags": ["wifi"]}'


In [ ]:
# Two ways a POST can be refused, with different status codes and different fixes.
r = requests.post(f"{BASE}/api/devices", data="this is not JSON", timeout=5)
print(r.status_code, r.json())                       # 400: the request could not even be parsed

r = requests.post(f"{BASE}/api/devices", json={"hostname": "incomplete"}, timeout=5)
print(r.status_code, r.json())                       # 422: parsed fine, but fields are missing

400 {'error': 'request body is not valid JSON', 'status': 400}
422 {'error': 'missing fields: ip, type', 'status': 422}


Notice that the API's *error body* explains what is wrong. When a call fails, **print the body** (`r.text`) before anything else. It saves hours.

### 7.6 `PUT`, `PATCH`, `DELETE`: the whole lifecycle

> **REFERENCE** Read it on your own; the lab practises it.

Using the device we just created, here is the full create, read, update, delete cycle. Note the difference between `PUT` (send the *whole* record, missing fields are lost or rejected) and `PATCH` (send only what changes).

In [ ]:
new_id = requests.post(f"{BASE}/api/devices", json=new_device, timeout=5).json()["id"]
url = f"{BASE}/api/devices/{new_id}"
print("created id", new_id)

print("\n-- READ")
print(requests.get(url, timeout=5).json())

print("\n-- PATCH: change one field")
r = requests.patch(url, json={"status": "down"}, timeout=5)
print(r.status_code, r.json())

print("\n-- PUT: replace the record (must send all required fields)")
r = requests.put(url, json={"hostname": "OTT-AP-09", "ip": "10.10.3.99", "type": "access-point"}, timeout=5)
print(r.status_code, r.json())

print("\n-- PUT with a missing field is refused")
r = requests.put(url, json={"hostname": "OTT-AP-09"}, timeout=5)
print(r.status_code, r.json())

print("\n-- DELETE")
r = requests.delete(url, timeout=5)
print(r.status_code, repr(r.text), "(204 means success with no body)")

print("\n-- READ again")
print(requests.get(url, timeout=5).status_code)

created id 13

-- READ
{'id': 13, 'hostname': 'OTT-AP-09', 'ip': '10.10.3.9', 'type': 'access-point', 'ports': 2, 'status': 'up', 'tags': ['wifi']}

-- PATCH: change one field
200 {'id': 13, 'hostname': 'OTT-AP-09', 'ip': '10.10.3.9', 'type': 'access-point', 'ports': 2, 'status': 'down', 'tags': ['wifi']}

-- PUT: replace the record (must send all required fields)
200 {'id': 13, 'hostname': 'OTT-AP-09', 'ip': '10.10.3.99', 'type': 'access-point', 'ports': 0, 'status': 'up', 'tags': []}

-- PUT with a missing field is refused
422 {'error': 'PUT replaces the whole record; missing: ip, type', 'status': 422}

-- DELETE
204 '' (204 means success with no body)

-- READ again
404


### 7.7 Headers and content negotiation

> **REFERENCE** Read it on your own; the lab practises it.

Set request headers with `headers=`. The most useful is `Accept`, which tells the server which format you prefer. Our practice API offers the same inventory in three formats on three URLs (many real APIs use one URL and the `Accept` header instead). It is the section 2 to 4 material reappearing: **the format is just the body, and you already know how to parse each one.**

In [ ]:
reset_api()
as_json = requests.get(f"{BASE}/api/devices", params={"per_page": 50}, timeout=5).json()["results"]

xml_reply = requests.get(f"{BASE}/api/devices.xml", timeout=5)
yaml_reply = requests.get(f"{BASE}/api/devices.yaml", timeout=5)

print(xml_reply.headers["Content-Type"], "|", yaml_reply.headers["Content-Type"])

from_xml = ET.fromstring(xml_reply.text).findall("device")
from_yaml = yaml.safe_load(yaml_reply.text)["devices"]

print("JSON :", len(as_json), "devices, first =", as_json[0]["hostname"])
print("XML  :", len(from_xml), "devices, first =", from_xml[0].findtext("hostname"))
print("YAML :", len(from_yaml), "devices, first =", from_yaml[0]["hostname"])

# Same information, three parsers; the numbers are strings in XML only.
print(type(as_json[0]["ports"]).__name__, type(from_xml[0].findtext("ports")).__name__, type(from_yaml[0]["ports"]).__name__)

application/xml | application/yaml
JSON : 11 devices, first = OTT-CORE-01
XML  : 11 devices, first = OTT-CORE-01
YAML : 11 devices, first = OTT-CORE-01
int str int


In [ ]:
# Custom headers show up on the server. User-Agent identifies your tool to the API owner.
r = requests.get(f"{BASE}/api/echo", headers={"Accept": "application/json", "User-Agent": "bytown-netops/1.0"}, timeout=5)
print(r.json())
print("default User-Agent would have been:", requests.utils.default_user_agent())

{'query': {}, 'user_agent': 'bytown-netops/1.0', 'accept': 'application/json'}
default User-Agent would have been: python-requests/2.32.4


### 7.8 Authentication

> **REFERENCE** Read it on your own; the lab practises it.

Most real APIs require you to prove who you are. The patterns you will meet, from most to least common in DevOps:

| Scheme | How it is sent | In `requests` |
|---|---|---|
| **Bearer token** (API token, personal access token, OAuth) | `Authorization: Bearer <token>` | `headers={"Authorization": f"Bearer {token}"}` |
| **API key** | a custom header (`X-API-Key`) or a query parameter | `headers={...}` or `params={...}` |
| **Basic auth** | `Authorization: Basic base64(user:password)` | `auth=("user", "password")` |

The status codes tell you which half failed: **401** (no or unrecognized credentials) versus **403** (recognized, but not allowed).

**Secrets never go in your source code.** A token committed to Git is a leaked token (you will see why next week). Read it from an environment variable, or ask for it with `getpass` when running by hand.

In [ ]:
r = requests.get(f"{BASE}/api/secure/ping", timeout=5)
print("no credentials :", r.status_code, r.json())

r = requests.get(f"{BASE}/api/secure/ping", headers={"Authorization": "Bearer wrong-token"}, timeout=5)
print("wrong token    :", r.status_code, r.json())

no credentials : 401 {'error': 'missing Authorization header', 'status': 401}
wrong token    : 403 {'error': 'token not accepted', 'status': 403}


In [19]:
import os

# Pattern for real scripts: the secret comes from the environment, never from the source file.
os.environ.setdefault("NETOPS_TOKEN", "netops-token")        # for the demo only: normally set outside Python

token = os.environ["NETOPS_TOKEN"]                           # KeyError if missing, which is what you want
headers = {"Authorization": f"Bearer {token}"}

r = requests.get(f"{BASE}/api/secure/ping", headers=headers, timeout=5)
print("correct token  :", r.status_code, r.json())

correct token  : 200 {'pong': True, 'user': 'netops'}


### 7.9 Timeouts

> **REFERENCE** Read it on your own; the lab practises it.

By default **`requests` waits forever.** A server that accepts the connection and never answers will hang your script, and an unattended automation job that hangs is worse than one that fails. Set `timeout=` on **every** call.

- A single number is used for both phases: `timeout=5`.
- A tuple `(connect, read)` sets them separately: `timeout=(3.05, 10)`. *Connect* is how long to wait to reach the server; *read* is how long to wait between bytes of the response.

In [ ]:
import time

start = time.perf_counter()
try:
    requests.get(f"{BASE}/api/slow", timeout=0.5)            # the server takes 2 seconds
except requests.Timeout as e:
    print(f"gave up after {time.perf_counter() - start:.1f}s ->", type(e).__name__)

start = time.perf_counter()
r = requests.get(f"{BASE}/api/slow", timeout=5)
print(f"patient call succeeded after {time.perf_counter() - start:.1f}s ->", r.json())

gave up after 0.5s -> ReadTimeout
patient call succeeded after 2.0s -> {'ok': True, 'waited': 2}


### 7.10 The exception hierarchy

> **REFERENCE** Read it on your own; the lab practises it.

Network calls fail in ways that have nothing to do with HTTP status codes: the host is down, DNS fails, the connection drops. `requests` raises exceptions for those, all descended from `requests.RequestException`. This is Week 4's exception hierarchy applied to a library.

```
RequestException
 ├── ConnectionError        could not reach the server at all (down, refused, DNS)
 │    └── ConnectTimeout    ... and it timed out trying to connect
 ├── Timeout                the server was reached but too slow
 │    ├── ConnectTimeout
 │    └── ReadTimeout
 ├── HTTPError              raised by raise_for_status() on 4xx/5xx
 ├── TooManyRedirects
 └── JSONDecodeError        r.json() on a body that is not JSON
```

Order your `except` clauses from specific to general, and recall from Week 4 that the **first matching clause wins**.

In [ ]:
def check_url(url):
    """Classify what happened when we call a URL. One try block around the one risky line."""
    try:
        r = requests.get(url, timeout=1)
        r.raise_for_status()
    except requests.Timeout:
        return "timeout"
    except requests.ConnectionError:
        return "connection failed"
    except requests.HTTPError as e:
        return f"HTTP error {e.response.status_code}"
    except requests.RequestException as e:                      # catch-all for anything else in the family
        return f"other request failure: {type(e).__name__}"
    return "ok"

print("healthy endpoint :", check_url(f"{BASE}/api/devices"))
print("404              :", check_url(f"{BASE}/api/devices/999"))
print("slow endpoint    :", check_url(f"{BASE}/api/slow"))
print("nothing listening:", check_url("http://127.0.0.1:9/"))
print("unknown hostname :", check_url("http://no-such-host.invalid/"))

healthy endpoint : ok
404              : HTTP error 404
slow endpoint    : timeout
nothing listening: connection failed
unknown hostname : connection failed


In [ ]:
# r.json() fails when the body is not JSON, for example an HTML error page from a proxy in front of the API.
r = requests.get(f"{BASE}/api/devices.yaml", timeout=5)            # valid response, but YAML not JSON
try:
    r.json()
except requests.JSONDecodeError as e:
    print("JSONDecodeError:", e)
    print("Content-Type was :", r.headers["Content-Type"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)
Content-Type was : application/yaml


### 7.11 Sessions

> **REFERENCE** Read it on your own; the lab practises it.

Every bare `requests.get()` opens a new connection, does the TCP and TLS handshake, makes the call, and closes it. For a script making dozens of calls to the same host, that is slow. A **`requests.Session`** keeps connections open and lets you set things **once**: headers, authentication, and (with a little code) a default timeout.

In [ ]:
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}", "User-Agent": "bytown-netops/1.0", "Accept": "application/json"})

# No need to repeat the headers on each call.
print(session.get(f"{BASE}/api/secure/ping", timeout=5).json())
print(session.get(f"{BASE}/api/echo", timeout=5).json())

# Use as a context manager so connections are closed when you are done.
with requests.Session() as s:
    s.headers["Accept"] = "application/json"
    print(s.get(f"{BASE}/api/devices/1", timeout=5).json()["hostname"])

{'pong': True, 'user': 'netops'}
{'query': {}, 'user_agent': 'bytown-netops/1.0', 'accept': 'application/json'}
OTT-CORE-01


### 7.12 Retrying: when, and how

> **REFERENCE** Read it on your own; the lab practises it.

Some failures are **transient**: a 503 during a deployment, a dropped connection, a rate limit. Retrying helps. Others are **permanent**: a 404, a 422, a 401. Retrying those only wastes time and can annoy the owner of the API.

| Retry | Do not retry |
|---|---|
| connection errors and timeouts | 400, 401, 403, 404, 422 (your request is wrong) |
| 500, 502, 503, 504 | any non-idempotent `POST`, unless the API supports idempotency keys |
| 429, after waiting `Retry-After` | |

Three habits make retrying safe: a **limit** on attempts, a **delay** between them, and the delay should **grow** (exponential backoff: 1s, 2s, 4s, ...) so that you do not hammer a service that is struggling.

The `/api/flaky` endpoint returns 503 on the first two calls and succeeds on the third. Here is the hand-written loop first, then the Week 4 decorator version.

In [ ]:
requests.post(f"{BASE}/api/reset", timeout=5)               # resets the flaky counter

def get_with_retry(url, attempts=4, base_delay=0.2):
    """GET with exponential backoff. Retries only on transient failures."""
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(url, timeout=3)
            if r.status_code in (500, 502, 503, 504):
                raise requests.HTTPError(f"{r.status_code} from server", response=r)
            return r
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as e:
            if attempt == attempts:
                raise                                      # out of attempts: let the caller see the failure
            delay = base_delay * 2 ** (attempt - 1)
            print(f"  attempt {attempt} failed ({e}); waiting {delay:.1f}s")
            time.sleep(delay)

r = get_with_retry(f"{BASE}/api/flaky")
print("succeeded:", r.status_code, r.json())

  attempt 1 failed (503 from server); waiting 0.2s
  attempt 2 failed (503 from server); waiting 0.4s
succeeded: 200 {'ok': True, 'calls_needed': 3}


In [ ]:
# The same idea as a decorator: last week's @retry, now with a delay and the requests exceptions.
import functools

def retry(times=3, delay=0.2, exceptions=(requests.ConnectionError, requests.Timeout)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == times:
                        raise
                    print(f"  {func.__name__}: attempt {attempt} failed ({type(e).__name__}); retrying")
                    time.sleep(delay * 2 ** (attempt - 1))
        return wrapper
    return decorator

requests.post(f"{BASE}/api/reset", timeout=5)

@retry(times=4, delay=0.1, exceptions=(requests.HTTPError,))
def fetch_flaky():
    r = requests.get(f"{BASE}/api/flaky", timeout=3)
    r.raise_for_status()                    # turns 503 into HTTPError, which the decorator retries
    return r.json()

print(fetch_flaky())

  fetch_flaky: attempt 1 failed (HTTPError); retrying
  fetch_flaky: attempt 2 failed (HTTPError); retrying
{'ok': True, 'calls_needed': 3}


Rate limiting is a polite version of the same thing: the server tells you exactly how long to wait. Honour the `Retry-After` header rather than guessing.

In [ ]:
requests.post(f"{BASE}/api/reset", timeout=5)

results = []
for i in range(1, 7):
    r = requests.get(f"{BASE}/api/limited", timeout=3)
    if r.status_code == 429:
        wait = int(r.headers.get("Retry-After", "1"))
        print(f"call {i}: 429 rate limited, server says wait {wait}s")
        time.sleep(wait)
        r = requests.get(f"{BASE}/api/limited", timeout=3)         # one retry after waiting
    results.append(r.status_code)
    print(f"call {i}: final status {r.status_code}")

print(results)

call 1: final status 200
call 2: final status 200
call 3: 429 rate limited, server says wait 1s
call 3: final status 200
call 4: final status 200
call 5: 429 rate limited, server says wait 1s
call 5: final status 200
call 6: final status 200
[200, 200, 200, 200, 200, 200]


`requests` can also do this for you: mount an `HTTPAdapter` with a `urllib3.util.Retry` policy onto a session (`total`, `backoff_factor`, `status_forcelist`, `allowed_methods`). Production code often does. Writing the loop by hand once, as above, is how you learn what that policy is doing.

### 7.13 Pagination

> **REFERENCE** Read it on your own; the lab practises it.

A list endpoint that returns *every* record would be slow, huge, and dangerous at scale, so APIs return **a page at a time** and tell you how to get the next one. Three styles exist:

| Style | The response gives you | You send |
|---|---|---|
| **Page number** | `page`, `per_page`, `total` | `?page=2` |
| **Next link** | a `next` URL (in the body, or a `Link` header) | request that URL |
| **Cursor** | an opaque token | `?cursor=abc123` |

Our API returns both `total` and a ready-made `next` URL (or `null` on the last page). Following `next` until it disappears is the most robust loop because **you never compute anything**: the server decides.

Write it as a **generator** so the caller can process items as they arrive without caring about pages.

In [ ]:
reset_api()

def iter_devices(session, **filters):
    """Yield every device across all pages by following the server's own 'next' link."""
    url = f"{BASE}/api/devices"
    params = {"per_page": 4, **filters}
    while url:
        r = session.get(url, params=params, timeout=5)
        r.raise_for_status()
        page = r.json()
        yield from page["results"]
        url = page["next"]                # None on the last page ends the loop
        params = None                     # the next URL already contains its own query string

with requests.Session() as s:
    hostnames = [d["hostname"] for d in iter_devices(s)]
    print(len(hostnames), "devices:", hostnames)

    down = [d["hostname"] for d in iter_devices(s, status="down")]
    print("down:", down)

11 devices: ['OTT-CORE-01', 'OTT-CORE-02', 'OTT-SW-01', 'OTT-SW-02', 'OTT-SW-03', 'TOR-FW-01', 'TOR-FW-02', 'TOR-SW-01', 'MTL-RTR-01', 'MTL-SW-01', 'MTL-AP-01']
down: ['OTT-SW-02', 'TOR-FW-02', 'MTL-AP-01']


In [ ]:
# See the pages themselves, to understand what the generator is hiding.
with requests.Session() as s:
    url, page_no = f"{BASE}/api/devices?per_page=4", 0
    while url:
        page = s.get(url, timeout=5).json()
        page_no += 1
        print(f"page {page['page']}: {len(page['results'])} items | total={page['total']} | next={'yes' if page['next'] else 'no'}")
        url = page["next"]

page 1: 4 items | total=11 | next=yes
page 2: 4 items | total=11 | next=yes
page 3: 3 items | total=11 | next=no


> **Common pitfall.** Reading only the first page. A script that calls `GET /devices` once, finds 4 devices, and reports "4 devices" looks right and is wrong in a way no exception will ever tell you. Whenever a list endpoint has `next`, `total` or a `Link` header, you must loop.

### 7.14 A real public API

> **REFERENCE** Read it on your own; the lab practises it.

Everything above would work identically against a real server on the internet; only `BASE` changes. GitHub's public REST API needs no account for a few calls (rate limited to 60 per hour per IP), and it uses exactly the pieces you now know: JSON bodies, status codes, headers, and a rate limit.

The cell is wrapped so the notebook still runs when you are offline or the limit is used up.

In [ ]:
try:
    r = requests.get(
        "https://api.github.com/repos/psf/requests",
        headers={"Accept": "application/vnd.github+json", "User-Agent": "bytown-netops-demo"},
        timeout=5,
    )
    r.raise_for_status()
    repo = r.json()
    print("repository  :", repo["full_name"])
    print("description :", repo["description"])
    print("language    :", repo["language"])
    print("rate limit  :", r.headers.get("X-RateLimit-Remaining"), "of", r.headers.get("X-RateLimit-Limit"), "calls left this hour")
except requests.RequestException as e:
    print("Could not reach the public API from here:", type(e).__name__)
    print("That is fine: the exception handling is the point. Continue with the practice API.")

repository  : psf/requests
description : A simple, yet elegant, HTTP library.
language    : Python
rate limit  : 59 of 60 calls left this hour


---
## 8. Putting it together: an inventory client

> **REFERENCE** Read it on your own; the lab practises it.

Now combine everything with Weeks 3 and 4: a **class** (OOP) that wraps the API, a **custom exception** (error handling), a **session** with defaults, a **generator** for pagination, and **file output** in all three formats.

Design decisions to notice:

- the class holds the session, the base URL, and the timeout, so no method repeats them;
- every HTTP failure becomes one exception type, `InventoryAPIError`, which carries the status code. The rest of the program catches **one** thing and does not need to know about `requests`;
- methods return plain dicts, not `Response` objects, so callers do not depend on the HTTP library.

In [20]:
class InventoryAPIError(Exception):
    """Raised when the inventory API fails or refuses a request."""
    def __init__(self, message, status_code=None):
        super().__init__(message)
        self.status_code = status_code


class InventoryClient:
    def __init__(self, base_url, token=None, timeout=5):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers["Accept"] = "application/json"
        if token:
            self.session.headers["Authorization"] = f"Bearer {token}"

    # ----- one place that makes requests, so error handling is written once -----
    def _request(self, method, path, **kwargs):
        url = f"{self.base_url}{path}"
        try:
            r = self.session.request(method, url, timeout=self.timeout, **kwargs)
        except requests.Timeout as e:
            raise InventoryAPIError(f"{method} {path} timed out after {self.timeout}s") from e
        except requests.ConnectionError as e:
            raise InventoryAPIError(f"cannot reach {self.base_url}") from e

        if not r.ok:
            try:
                detail = r.json().get("error", r.text)
            except ValueError:
                detail = r.text
            raise InventoryAPIError(f"{method} {path} -> {r.status_code}: {detail}", r.status_code)
        return r.json() if r.content else None

    # ----- the API, as Python methods -----
    def get(self, device_id):
        return self._request("GET", f"/api/devices/{device_id}")

    def devices(self, **filters):
        """Generator: yields every device across all pages."""
        path, params = "/api/devices", {"per_page": 5, **filters}
        while True:
            page = self._request("GET", path, params=params)
            yield from page["results"]
            if not page["next"]:
                return
            params = {**params, "page": page["page"] + 1}

    def create(self, **fields):
        return self._request("POST", "/api/devices", json=fields)

    def update(self, device_id, **changes):
        return self._request("PATCH", f"/api/devices/{device_id}", json=changes)

    def delete(self, device_id):
        self._request("DELETE", f"/api/devices/{device_id}")

    def ping(self):
        return self._request("GET", "/api/secure/ping")


reset_api()
client = InventoryClient(BASE, token=os.environ["NETOPS_TOKEN"])
print(client.ping())
print(client.get(1)["hostname"])

{'pong': True, 'user': 'netops'}
OTT-CORE-01


In [ ]:
# Errors arrive as ONE exception type with the information the caller needs.
for action, call in [
    ("missing device", lambda: client.get(999)),
    ("invalid create", lambda: client.create(hostname="incomplete")),
    ("bad token",      lambda: InventoryClient(BASE, token="nope").ping()),
    ("unreachable",    lambda: InventoryClient("http://127.0.0.1:9", timeout=1).get(1)),
]:
    try:
        call()
    except InventoryAPIError as e:
        print(f"{action:<15} status={e.status_code}  {e}")

missing device  status=404  GET /api/devices/999 -> 404: device 999 not found
invalid create  status=422  POST /api/devices -> 422: missing fields: ip, type
bad token       status=403  GET /api/secure/ping -> 403: token not accepted
unreachable     status=None  cannot reach http://127.0.0.1:9


---
## Wrap-up

Weeks 3 and 4 gave you the language and the habits. This session connected that language to the outside world:

- **Formats** are agreements. JSON is the default for APIs; XML is what many devices and enterprise tools speak; YAML is for configuration people edit. In Python all three become the dicts, lists, strings and numbers you already know, and `json`, `ElementTree` and `PyYAML` follow the same load/dump pattern.
- **Parsing succeeding does not mean the data is right.** Check the shape. XML text is always a string; YAML guesses types; JSON turns tuples into lists and integer keys into strings.
- **HTTP** is a request (method, URL, headers, body) and a response (status, headers, body). The status class tells you whose fault it is and whether to retry.
- **`requests`** gives you one function per method. Remember what it does *not* do for you: it does not raise on 4xx/5xx (call `raise_for_status()`), it does not time out (always pass `timeout=`), and it does not follow pages (loop on `next`).
- **Wrap the API in your own class** so that errors, defaults and authentication live in one place.

**Seven habits to carry forward**

1. Always `yaml.safe_load`, never `yaml.load`.
2. Always pass `timeout=`.
3. Print `r.text` before you debug anything else.
4. Quote YAML values that must stay strings.
5. Convert XML text to the type you need before using it.
6. Retry only transient, idempotent failures, with a limit and a growing delay.
7. Secrets come from the environment, never from the source file.

**What the lab expects.** The Week 5 lab runs this same practice API and walks through each section again with more examples, then asks you to complete graded tasks on JSON, XML, YAML, `requests` CRUD, authentication, retries and pagination. **Practical Assessment 1** is a separate `.py` file that covers the Python from Weeks 3 and 4 (it does not need the network).

**Next week:** Version control with Git and GitHub. GitHub is itself a REST API, so everything in sections 6 and 7 applies to it directly.

In [ ]:
# Housekeeping: stop the practice API and remove the demo files this notebook created.
import shutil
mockapi.stop()
shutil.rmtree(out_dir, ignore_errors=True)
print("practice API stopped; demo folder removed")

practice API stopped; demo folder removed
